# 🎯 Technique 82: Temperature Tuning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/10-optimization/82_temperature_tuning.ipynb)

**Category:** 10 - Optimization & Auto-Tuning  
**Technique #:** 82  
**Difficulty:** Beginner

## 📋 Description

Temperature controls the randomness of LLM outputs. Lower temperatures produce more deterministic responses, while higher temperatures produce more creative, diverse outputs.

**When to use:**
- Need deterministic outputs (low temp)
- Want creative responses (high temp)
- Balancing accuracy with diversity

## 🔧 How It Works

Temperature modifies logits before softmax:
- T=0.0: Always picks most likely token (greedy)
- T=0.7: Balanced (default)
- T=1.0+: More random, creative

## ⚙️ Setup

In [ ]:
!pip install -q openai matplotlib
import openai
import matplotlib.pyplot as plt
import numpy as np
from typing import List, Dict
from getpass import getpass

In [ ]:
openai.api_key = getpass('Enter your OpenAI API key: ')

## 🛠️ Implementation: Temperature Explorer

In [ ]:
class TempExplorer:
    def __init__(self, model='gpt-4o-mini'):
        self.model = model
    
    def generate(self, prompt, temperature, n=1, max_tokens=150):
        responses = []
        for _ in range(n):
            response = openai.chat.completions.create(
                model=self.model,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=temperature,
                max_tokens=max_tokens
            )
            responses.append(response.choices[0].message.content.strip())
        return responses
    
    def compare(self, prompt, temperatures, samples_per=3):
        results = {}
        for temp in temperatures:
            print(f'Generating at T={temp}...')
            results[temp] = self.generate(prompt, temp, n=samples_per)
        return results
    
    def visualize(self):
        logits = np.array([2.0, 1.5, 1.0, 0.5, 0.3, 0.2, 0.1, 0.05])
        temps = [0.3, 0.7, 1.0, 1.5]
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        axes = axes.flatten()
        for i, T in enumerate(temps):
            scaled = logits / T
            exp = np.exp(scaled - np.max(scaled))
            probs = exp / exp.sum()
            axes[i].bar(range(len(probs)), probs)
            axes[i].set_title(f'Temperature = {T}')
            axes[i].set_ylim(0, 1)
        plt.tight_layout()
        plt.show()

## 💡 Basic Example: Temperature Comparison

In [ ]:
explorer = TempExplorer()
prompt = 'Write a creative opening line for a mystery novel.'

print('Temperature Comparison\n')
results = explorer.compare(prompt, [0.0, 0.5, 1.0], samples_per=2)

for temp, responses in results.items():
    print(f'\nT={temp}:')
    for i, r in enumerate(responses, 1):
        print(f'  {i}. {r}')

In [ ]:
print('\nVisualizing temperature effect on probability distribution:\n')
explorer.visualize()

## 🌍 Real-World Example: Task-Specific Temperature Selection

In [ ]:
tasks = [
    {'name': 'Code Generation', 'prompt': 'Write a Python function for factorial.', 'temp': 0.2},
    {'name': 'Sentiment Analysis', 'prompt': 'Classify: I love this product!', 'temp': 0.1},
    {'name': 'Customer Support', 'prompt': 'Respond to: My order is late.', 'temp': 0.7},
    {'name': 'Brainstorming', 'prompt': 'List 5 marketing ideas for coffee.', 'temp': 0.9}
]

for task in tasks:
    print(f'\n{task["name"]} (T={task["temp"]}):')
    response = explorer.generate(task['prompt'], task['temp'], max_tokens=50)[0]
    print(f'  {response[:80]}...')

## ⚠️ Failure Case: Wrong Temperature Selection

In [ ]:
print('Wrong Temperature Examples:\n')

# High temp for code
code_prompt = 'Write a Python function to check if prime. Return only code.'
print('High temp (1.0) for code:')
print(explorer.generate(code_prompt, 1.0, max_tokens=50)[0][:60])
print('\nLow temp (0.2) for code:')
print(explorer.generate(code_prompt, 0.2, max_tokens=50)[0][:60])

print('\n---\n')

# Low temp for creative
creative = 'Write a unique tagline for coffee.'
print('Low temp (0.1) for creative:')
for i in range(2):
    print(f'  {explorer.generate(creative, 0.1, max_tokens=30)[0]}')
print('\nHigh temp (0.9) for creative:')
for i in range(2):
    print(f'  {explorer.generate(creative, 0.9, max_tokens=30)[0]}')

## 📊 Temperature Benchmarks

| Task Type | Recommended T | Consistency | Creativity |
|-----------|--------------:|-------------|------------|\n| Code | 0.0-0.2 | High | Low |\n| Classification | 0.1-0.3 | High | Low |\n| Q&A | 0.3-0.5 | Medium | Medium |\n| Chat | 0.6-0.8 | Medium | High |\n| Creative | 0.8-1.0 | Low | High |

## 🎮 Interactive Playground

In [ ]:
YOUR_PROMPT = 'Your prompt here'
YOUR_TEMPS = [0.0, 0.3, 0.7, 1.0]

# my_explorer = TempExplorer()
# results = my_explorer.compare(YOUR_PROMPT, YOUR_TEMPS, samples_per=2)
# for temp, responses in results.items():
#     print(f'T={temp}: {responses}')

## 💡 Tips & Tricks

- Start at 0.7 and adjust
- Lower for production (consistency)
- Higher for exploration (diversity)
- Use 0 for testing (reproducible)

## 📚 References

1. [OpenAI API Temperature](https://platform.openai.com/docs/api-reference/chat/create#chat-create-temperature)
2. [Temperature in Language Models](https://lukesalamone.github.io/posts/what-is-temperature/)